# Create, update, ACL, and delete a spec

`/admin/specs` stands in for product staff. A successful write posts a pointer
at the webhook receiver. The product connector fetches the markdown and writes
`raw/product/spec/{id}.md`. A lost pointer is recovered from `/ids`.

The only native ACL is `product_org`. Repeating it does not move the version.
Any other value is rejected and does not notify.

Requires `make up`.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "source_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import source_demo
product = source_demo.product


In [ ]:
SPEC_ID = "spec-demo-notebook"
KEY = product.key(SPEC_ID, "md")
BODY = "# Notebook spec\n\n## Scope\n\nCreated by the notebook.\n"
PAYLOAD = {
    "spec_id": SPEC_ID,
    "title": "Notebook spec",
    "product_key": "checkout",
    "body": BODY,
    "native_acl": ["product_org"],
}
product.delete_if_present("specs", SPEC_ID)
created = product.admin("POST", "/admin/specs", PAYLOAD)
print(created["spec_id"], "version", created["version"])
product.expect_status("POST", "/admin/specs", 409, PAYLOAD)
product.expect_status(
    "POST", "/admin/specs", 422, {**PAYLOAD, "spec_id": "nope", "native_acl": ["sales_leadership"]}
)


In [ ]:
def written():
    body = source_demo.object_body(KEY) if source_demo.snapshot(KEY) else b""
    return b"Created by the notebook." in body

source_demo.wait_until(written, source_demo.WRITE_WAIT, "spec snapshot")
print("snapshot", KEY, "chunks will split on the headings")


In [ ]:
edited = product.admin(
    "PUT",
    f"/admin/specs/{SPEC_ID}",
    {"body": "# Notebook spec\n\n## Scope\n\nEdited by the notebook.\n"},
)
print("version", edited["version"])

def changed():
    return b"Edited by the notebook." in source_demo.object_body(KEY)

source_demo.wait_until(changed, source_demo.WRITE_WAIT, "spec edit")


In [ ]:
before = product.admin("GET", f"/specs/{SPEC_ID}")["version"]
same = product.admin("PUT", f"/admin/specs/{SPEC_ID}/acl", {"native_acl": ["product_org"]})
print("version", same["version"], "unchanged" if same["version"] == before else "moved")
if same["version"] != before:
    raise SystemExit("repeating product_org must not move the version")


In [ ]:
product.admin("DELETE", f"/admin/specs/{SPEC_ID}")
source_demo.wait_until(
    lambda: source_demo.snapshot(KEY) is None, source_demo.GONE_WAIT, "spec deleted"
)
print("deleted", SPEC_ID)
